Alohan'ny mamerina dia avereno atao Run ny notebook iray manontolo. Ny fanaovana azy dia redémarrena mihitsy ny kernel aloha (jereo menubar, safidio **Kernel$\rightarrow$Restart Kernel and Run All Cells**).

Izay misy hoe `YOUR CODE HERE` na "YOUR ANSWER HERE" ihany no fenoina. Afaka manampy cells vaovao raha ilaina. Aza adino ny mameno references eo ambany raha ilaina.

## References
Eto ilay references rehetra no apetraka

---

In [1]:
import numpy as np
import scipy
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
import cvxpy as cp
from sklearn.linear_model import Lasso
import warnings
warnings.filterwarnings("ignore")
from random import randrange
from sklearn.datasets import load_boston, load_diabetes, load_iris, load_digits, load_breast_cancer, make_blobs
from sklearn.model_selection import train_test_split

def grad_check_sparse(f, x, analytic_grad, num_checks=12, h=1e-5, error=1e-9):
    """
    sample a few random elements and only return numerical
    in this dimensions
    """

    for i in range(num_checks):
        ix = tuple([randrange(m) for m in x.shape])

        oldval = x[ix]
        x[ix] = oldval + h  # increment by h
        fxph = f(x)  # evaluate f(x + h)
        x[ix] = oldval - h  # increment by h
        fxmh = f(x)  # evaluate f(x - h)
        x[ix] = oldval  # reset

        grad_numerical = (fxph - fxmh) / (2 * h)
        grad_analytic = analytic_grad[ix]
        rel_error = abs(grad_numerical - grad_analytic) / (
            abs(grad_numerical) + abs(grad_analytic)
        )
        print(
            "numerical: %f analytic: %f, relative error: %e"
            % (grad_numerical, grad_analytic, rel_error)
        )
        assert rel_error < error

def rel_error(x, y):
    """ returns relative error """
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

# Alternative Linear regression

Implement linear regression using the following alternative loss instead of MSE:

$$Loss(\mathbf{w})= \frac{1}{N} \sum_{i=1}^N h_\epsilon(\mathbf{w}^\top\mathbf{x}_i +b - \mathbf{y}_i) + \lambda ||\mathbf{w}||^2_2$$

where $$h_\epsilon(r) =  \begin{cases}
    r^2/2 & \text{if } |r|\le \epsilon \\ 
    \epsilon|r|-\epsilon^2/2 & \text{if } |r| \gt \epsilon
\end{cases}$$



In [2]:
data = load_boston()
X_train1, y_train1 = data.data, data.target
w1 = np.random.randn(X_train1.shape[1]) * 0.0001
b1 = np.random.randn(1) * 0.0001

In [3]:
#Cell vaovao
#Fonction nampiana
###Begin Solution
def h_epsilon(r, epsilon):
    return np.where(np.abs(r) <= epsilon, 0.5 * r**2, epsilon * np.abs(r) - 0.5 * epsilon**2)

def h_epsilon_prime(r, epsilon):
    return np.where(np.abs(r) <= epsilon, r, epsilon * np.sign(r))
###End solution

In [4]:
def alternative_loss_lr_naive(w, b, X, y, epsilon=1.35, alpha=0.0001):
    """
    Naive loss for all observations
   
    Inputs:
    - w: array of shape (D,) containing weights
    - b: float bias 
    - X: array of shape (N, D) containing a minibatch of data
    - y: array of shape (N,) containing training labels 
    - epsilon: float
    - alpha: regularization
    """
    ### Begin solution
    loss = 0.0
    dw = np.zeros_like(w)
    db = 0.0
    N = X.shape[0]
    for i in range(N):
        prediction = np.dot(X[i], w) + b
        loss += h_epsilon(prediction - y[i], epsilon)
        dw += h_epsilon_prime(prediction - y[i], epsilon) * X[i]

    loss = (1/N) * loss + alpha * np.sum(w**2)
    dw = (1/N) * dw + 2 * alpha * w
    db = (1/N) * np.sum(h_epsilon_prime(np.dot(X, w) + b - y, epsilon))
    return loss, dw, np.array(db).reshape(1,)
    ### End solution

## without regularization

In [5]:
loss, dw1, db1 = alternative_loss_lr_naive(w1, b1, X_train1, y_train1, epsilon=1.35, alpha=0)

print("Gradient check w")
# Check with numerical gradient w
f = lambda w1: alternative_loss_lr_naive(w1, b1, X_train1, y_train1, epsilon=1.35, alpha=0)[0]
grad_numerical = grad_check_sparse(f, w1, dw1, 15, error=1e-8)

print("Gradient check bias")
# Check with numerical gradient b
f2 = lambda b1: alternative_loss_lr_naive(w1, b1, X_train1, y_train1, epsilon=1.35, alpha=0)[0]
grad_numerical = grad_check_sparse(f2, b1, db1, 15, error=1e-8)


# Large epsilon
large_eps_loss, large_eps_dw1, large_eps_db1 = alternative_loss_lr_naive(w1, b1, X_train1, y_train1, epsilon=135, alpha=0)

print("Gradient check w large epsilon")
# Check with numerical gradient w
f = lambda w1: alternative_loss_lr_naive(w1, b1, X_train1, y_train1, epsilon=135, alpha=0)[0]
grad_numerical = grad_check_sparse(f, w1, large_eps_dw1, 15, error=1e-8)

print("Gradient check bias large epsilon")
# Check with numerical gradient b
f2 = lambda b1: alternative_loss_lr_naive(w1, b1, X_train1, y_train1, epsilon=135, alpha=0)[0]
grad_numerical = grad_check_sparse(f2, b1, large_eps_db1, 15, error=1e-8)

Gradient check w
numerical: -0.748838 analytic: -0.748838, relative error: 8.490450e-10
numerical: -17.081635 analytic: -17.081635, relative error: 2.629880e-11
numerical: -12.891700 analytic: -12.891700, relative error: 2.147110e-11
numerical: -5.123308 analytic: -5.123308, relative error: 1.220051e-10
numerical: -0.748838 analytic: -0.748838, relative error: 8.490450e-10
numerical: -8.484256 analytic: -8.484256, relative error: 2.198434e-10
numerical: -17.081635 analytic: -17.081635, relative error: 2.629880e-11
numerical: -15.340909 analytic: -15.340909, relative error: 1.861185e-12
numerical: -15.034651 analytic: -15.034651, relative error: 1.515066e-10
numerical: -17.081635 analytic: -17.081635, relative error: 2.629880e-11
numerical: -551.120158 analytic: -551.120158, relative error: 1.126203e-12
numerical: -92.576117 analytic: -92.576117, relative error: 8.240815e-12
numerical: -12.891700 analytic: -12.891700, relative error: 2.147110e-11
numerical: -15.034651 analytic: -15.0346

 ## with regularization

In [6]:
loss, dw1, db1 = alternative_loss_lr_naive(w1, b1, X_train1, y_train1, epsilon=1.35, alpha=1)

print("Gradient check w")
# Check with numerical gradient w
f = lambda w1: alternative_loss_lr_naive(w1, b1, X_train1, y_train1, epsilon=1.35, alpha=1)[0]
grad_numerical = grad_check_sparse(f, w1, dw1, 15, error=1e-8)

print("Gradient check bias")
# Check with numerical gradient b
f2 = lambda b1: alternative_loss_lr_naive(w1, b1, X_train1, y_train1, epsilon=1.35, alpha=1)[0]
grad_numerical = grad_check_sparse(f2, b1, db1, 15, error=1e-8)


# Large epsilon
large_eps_loss, large_eps_dw1, large_eps_db1 = alternative_loss_lr_naive(w1, b1, X_train1, y_train1, epsilon=135, alpha=1)

print("Gradient check w large epsilon")
# Check with numerical gradient w
f = lambda w1: alternative_loss_lr_naive(w1, b1, X_train1, y_train1, epsilon=135, alpha=1)[0]
grad_numerical = grad_check_sparse(f, w1, large_eps_dw1, 15, error=1e-8)

print("Gradient check bias large epsilon")
# Check with numerical gradient b
f2 = lambda b1: alternative_loss_lr_naive(w1, b1, X_train1, y_train1, epsilon=135, alpha=1)[0]
grad_numerical = grad_check_sparse(f2, b1, large_eps_db1, 15, error=1e-8)

Gradient check w
numerical: -15.034832 analytic: -15.034832, relative error: 1.501526e-10
numerical: -92.576256 analytic: -92.576256, relative error: 7.723877e-12
numerical: -17.081637 analytic: -17.081637, relative error: 2.780730e-11
numerical: -0.748621 analytic: -0.748621, relative error: 8.699353e-10
numerical: -0.093398 analytic: -0.093398, relative error: 7.425896e-10
numerical: -8.484685 analytic: -8.484685, relative error: 2.201881e-10
numerical: -15.341075 analytic: -15.341075, relative error: 5.186506e-12
numerical: -0.093398 analytic: -0.093398, relative error: 7.425896e-10
numerical: -8.484685 analytic: -8.484685, relative error: 2.201881e-10
numerical: -5.123209 analytic: -5.123209, relative error: 1.371460e-10
numerical: -4.878147 analytic: -4.878147, relative error: 1.473038e-10
numerical: -8.484685 analytic: -8.484685, relative error: 2.201881e-10
numerical: -12.892018 analytic: -12.892018, relative error: 1.545182e-11
numerical: -24.915191 analytic: -24.915191, relati

In [7]:
def alternative_loss_lr_vectorized(w, b, X, y, epsilon=1.35, alpha=0.0001):
    """
    Vectorized for all observations
    
    Inputs:
    - w: array of shape (D,) containing weights
    - b: float bias 
    - X: array of shape (N, D) containing a minibatch of data
    - y: array of shape (N,) containing training labels 
    - epsilon: float
    - alpha: regularization
    """
    ### Begin solution
    loss = 0.0
    dw = np.zeros_like(w)
    db = 0
    N = X.shape[0]
    predictions = X.dot(w) + b
    loss = (1/N) * np.sum(h_epsilon(predictions - y, epsilon)) + alpha * np.sum(w**2)
    dw = (1/N) * X.T.dot(h_epsilon_prime(predictions - y, epsilon)) + 2 * alpha * w
    db = (1/N) * np.sum(h_epsilon_prime(predictions - y, epsilon))
    return loss, dw, np.array(db).reshape(1,)
    ### End solution

## without regularization

In [8]:
loss, dw1, db1 = alternative_loss_lr_vectorized(w1, b1, X_train1, y_train1, epsilon=1.35, alpha=0)

print("Gradient check w")
# Check with numerical gradient w
f = lambda w1: alternative_loss_lr_vectorized(w1, b1, X_train1, y_train1, epsilon=1.35, alpha=0)[0]
grad_numerical = grad_check_sparse(f, w1, dw1, 15, error=1e-8)

print("Gradient check bias")
# Check with numerical gradient b
f2 = lambda b1: alternative_loss_lr_vectorized(w1, b1, X_train1, y_train1, epsilon=1.35, alpha=0)[0]
grad_numerical = grad_check_sparse(f2, b1, db1, 15, error=1e-8)


# Large epsilon
large_eps_loss, large_eps_dw1, large_eps_db1 = alternative_loss_lr_vectorized(w1, b1, X_train1, y_train1, epsilon=135, alpha=0)

print("Gradient check w large epsilon")
# Check with numerical gradient w
f = lambda w1: alternative_loss_lr_vectorized(w1, b1, X_train1, y_train1, epsilon=135, alpha=0)[0]
grad_numerical = grad_check_sparse(f, w1, large_eps_dw1, 15, error=1e-8)

print("Gradient check bias large epsilon")
# Check with numerical gradient b
f2 = lambda b1: alternative_loss_lr_vectorized(w1, b1, X_train1, y_train1, epsilon=135, alpha=0)[0]
grad_numerical = grad_check_sparse(f2, b1, large_eps_db1, 15, error=1e-8)

Gradient check w
numerical: -92.576117 analytic: -92.576117, relative error: 3.931251e-13
numerical: -551.120158 analytic: -551.120158, relative error: 1.634794e-13
numerical: -15.340909 analytic: -15.340909, relative error: 1.861069e-12
numerical: -0.093379 analytic: -0.093379, relative error: 1.553476e-09
numerical: -5.123308 analytic: -5.123308, relative error: 1.668316e-11
numerical: -15.340909 analytic: -15.340909, relative error: 1.861069e-12
numerical: -0.748838 analytic: -0.748838, relative error: 1.373994e-10
numerical: -15.034651 analytic: -15.034651, relative error: 3.821471e-12
numerical: -15.340909 analytic: -15.340909, relative error: 1.861069e-12
numerical: -4.878257 analytic: -4.878257, relative error: 3.136281e-11
numerical: -12.891700 analytic: -12.891700, relative error: 8.040090e-13
numerical: -24.914970 analytic: -24.914970, relative error: 1.863056e-12
numerical: -15.034651 analytic: -15.034651, relative error: 3.821471e-12
numerical: -551.120158 analytic: -551.12

## with regularization

In [9]:
loss, dw1, db1 = alternative_loss_lr_vectorized(w1, b1, X_train1, y_train1, epsilon=1.35, alpha=1)

print("Gradient check w")
# Check with numerical gradient w
f = lambda w1: alternative_loss_lr_vectorized(w1, b1, X_train1, y_train1, epsilon=1.35, alpha=1)[0]
grad_numerical = grad_check_sparse(f, w1, dw1, 15, error=1e-8)

print("Gradient check bias")
# Check with numerical gradient b
f2 = lambda b1: alternative_loss_lr_vectorized(w1, b1, X_train1, y_train1, epsilon=1.35, alpha=1)[0]
grad_numerical = grad_check_sparse(f2, b1, db1, 15, error=1e-8)


# Large epsilon
large_eps_loss, large_eps_dw1, large_eps_db1 = alternative_loss_lr_vectorized(w1, b1, X_train1, y_train1, epsilon=135, alpha=1)

print("Gradient check w large epsilon")
# Check with numerical gradient w
f = lambda w1: alternative_loss_lr_vectorized(w1, b1, X_train1, y_train1, epsilon=135, alpha=1)[0]
grad_numerical = grad_check_sparse(f, w1, large_eps_dw1, 15, error=1e-8)

print("Gradient check bias large epsilon")
# Check with numerical gradient b
f2 = lambda b1: alternative_loss_lr_vectorized(w1, b1, X_train1, y_train1, epsilon=135, alpha=1)[0]
grad_numerical = grad_check_sparse(f2, b1, large_eps_db1, 15, error=1e-8)

Gradient check w
numerical: -481.509805 analytic: -481.509805, relative error: 2.646736e-13
numerical: -0.093398 analytic: -0.093398, relative error: 1.159328e-09
numerical: -0.093398 analytic: -0.093398, relative error: 1.159328e-09
numerical: -8.484685 analytic: -8.484685, relative error: 1.082812e-11
numerical: -92.576256 analytic: -92.576256, relative error: 9.100503e-13
numerical: -0.093398 analytic: -0.093398, relative error: 1.159328e-09
numerical: -12.892018 analytic: -12.892018, relative error: 5.214769e-12
numerical: -12.892018 analytic: -12.892018, relative error: 5.214769e-12
numerical: -15.341075 analytic: -15.341075, relative error: 5.186390e-12
numerical: -15.034832 analytic: -15.034832, relative error: 2.469205e-12
numerical: -551.120011 analytic: -551.120011, relative error: 2.084492e-13
numerical: -481.509805 analytic: -481.509805, relative error: 2.646736e-13
numerical: -17.081637 analytic: -17.081637, relative error: 1.809256e-12
numerical: -15.034832 analytic: -15.

In [10]:
class LinearModelRegression():
    def __init__(self):
        self.w = None
        self.b = None

    def train(self, X, y, learning_rate=1e-3, alpha=0.0001, num_iters=100, batch_size=200, verbose=False):
        N, d = X.shape
        
        if self.w is None: # Initialization
            self.w = 0.001 * np.random.randn(d)
            self.b = 0.0

        # Run stochastic gradient descent to optimize w
        
        loss_history = []
        for it in range(num_iters):
            X_batch = None
            y_batch = None
                                                               
            # Sample batch_size elements in X_batch and y_batch
            indices = np.random.choice(N, batch_size)
            X_batch = X[indices]
            y_batch = y[indices]
            
            # evaluate loss and gradient
            loss, dw, db = self.loss(X_batch, y_batch, alpha)
            loss_history.append(loss)

            # perform parameter update                                                                
            # Update the weights w and bias b using the gradient and the learning rate. 
            ### Begin solution
            self.w -= learning_rate * (dw + 2 * alpha * self.w)
            self.b -= learning_rate * db
            ###End solution
            
            if verbose and it % 10000 == 0:
                print("iteration %d / %d: loss %f" % (it, num_iters, loss))
                
        return loss_history

    def predict(self, X):
        pass

    def loss(self, X_batch, y_batch, reg):
        pass

class AlternativeLinearRegression(LinearModelRegression):
    """ Linear regression """

    def loss(self, X_batch, y_batch, alpha):
        return alternative_loss_lr_vectorized(self.w, self.b, X_batch, y_batch, alpha=alpha)
    
    def predict(self, X):
        ### Begin solution
        return X.dot(self.w) + self.b
        ### End solution

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train1 = scaler.fit_transform(X_train1)

model = AlternativeLinearRegression()
model.train(X_train1, y_train1, num_iters=75000, batch_size=64, learning_rate=1e-2, verbose=True)
pred = model.predict(X_train1)
mse = mean_squared_error(pred, y_train1)

print("MSE gradient descent model :", mse)
assert mse < 25

iteration 0 / 75000: loss 28.824970
iteration 10000 / 75000: loss 2.670979
iteration 20000 / 75000: loss 3.931515
iteration 30000 / 75000: loss 3.321423
iteration 40000 / 75000: loss 4.577995
iteration 50000 / 75000: loss 2.951729
iteration 60000 / 75000: loss 3.333295
iteration 70000 / 75000: loss 3.247330
MSE gradient descent model : 24.3019649705328


# Alternative multiclass classification

Implement classification using the following loss:

$$L(\mathbf{W}) = \sum_{i=1}^N \sum_{j \neq y_i} \max(0, s_j - s_{y_i} + 1) + \lambda||\mathbf{w}||^2_2$$ <br>
$$\text{where } s_j = (f(\mathbf{x}_i;\mathbf{W}))_j = (\mathbf{W}\mathbf{x}_i)_j \text{ is the score for the j-th class}$$

In [12]:
data = load_iris()
X, y = data.data, data.target

W = np.random.randn(X.shape[1], 3) * 0.0001

In [13]:
def alternative_classification_loss_naive(W, X, y, alpha):
    """
    Multiclass Naive loss function WITH FOR LOOPS

    Inputs:
    - W: array of shape (D, C) containing weights
    - X: array of shape (N, D) containing a minibatch of data
    - y: array of shape (N,) containing training labels
    - alpha: (float) regularization 

    Returns a tuple of:
    - loss as single float
    - gradient with respect to weights W;  same shape as W
    """
    
    # Initialization
    ### Begin solution 
    loss = 0.0
    dW = np.zeros_like(W)
    num_train = X.shape[0]
    num_classes = W.shape[1]
    for i in range(num_train):
        scores = X[i].dot(W)
        correct_class_score = scores[y[i]]
        for j in range(num_classes):
            if j == y[i]:
                continue
            margin = scores[j] - correct_class_score + 1
            if margin > 0:
                loss += margin
                dW[:, j] += X[i]
                dW[:, y[i]] -= X[i]
    loss /= num_train
    dW /= num_train
    loss += 0.5 * alpha * np.sum(W * W)
    dW += alpha * W
    return loss, dW
    ### End solution

In [14]:
# NO REGLARIZATION
loss, dW = alternative_classification_loss_naive(W, X, y, 0.0)

f = lambda W: alternative_classification_loss_naive(W, X, y, 0.0)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-9)

numerical: 0.287333 analytic: 0.287333, relative error: 3.134432e-11
numerical: 0.287333 analytic: 0.287333, relative error: 3.134432e-11
numerical: 0.953333 analytic: 0.953333, relative error: 1.150012e-12
numerical: -0.826667 analytic: -0.826667, relative error: 1.472082e-11
numerical: -1.794000 analytic: -1.794000, relative error: 1.653576e-13
numerical: 0.837333 analytic: 0.837333, relative error: 3.514976e-11
numerical: -0.092667 analytic: -0.092667, relative error: 9.624413e-11
numerical: 2.296000 analytic: 2.296000, relative error: 3.549232e-13
numerical: -0.826667 analytic: -0.826667, relative error: 1.472082e-11
numerical: 0.837333 analytic: 0.837333, relative error: 3.514976e-11
numerical: -0.370667 analytic: -0.370667, relative error: 3.634128e-11
numerical: -1.794000 analytic: -1.794000, relative error: 1.653576e-13


In [15]:
# With REGLARIZATION
loss, dW = alternative_classification_loss_naive(W, X, y, 2)

f = lambda W: alternative_classification_loss_naive(W, X, y, 2)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-9)

numerical: -0.126752 analytic: -0.126752, relative error: 1.482734e-11
numerical: -0.501695 analytic: -0.501695, relative error: 7.129774e-12
numerical: -0.126752 analytic: -0.126752, relative error: 1.482734e-11
numerical: -0.501695 analytic: -0.501695, relative error: 7.129774e-12
numerical: 0.083435 analytic: 0.083435, relative error: 2.168126e-10
numerical: -0.501695 analytic: -0.501695, relative error: 7.129774e-12
numerical: 2.295972 analytic: 2.295972, relative error: 1.388086e-12
numerical: -0.826432 analytic: -0.826432, relative error: 1.868050e-11
numerical: 0.952759 analytic: 0.952759, relative error: 3.885481e-12
numerical: -0.092829 analytic: -0.092829, relative error: 6.532472e-11
numerical: -0.370820 analytic: -0.370820, relative error: 3.164654e-11
numerical: -0.501695 analytic: -0.501695, relative error: 7.129774e-12


In [16]:
def alternative_classification_loss_vectorized(W, X, y, alpha):
    """
    Multiclass vectorized loss function WITHOUT FOR LOOPS

    Inputs:
    - W: array of shape (D, C) containing weights
    - X: array of shape (N, D) containing a minibatch of data
    - y: array of shape (N,) containing training labels
    - alpha: (float) regularization 

    Returns a tuple of:
    - loss as single float
    - gradient with respect to weights W;  same shape as W
    """
    # Initialize the loss and gradient to zero.
    ### Begin Solution
    loss = 0.0
    dW = np.zeros_like(W)
    scores = X.dot(W)
    correct_class_scores = scores[np.arange(X.shape[0]), y]
    margins = np.maximum(0, scores - correct_class_scores[:, np.newaxis] + 1)
    margins[np.arange(X.shape[0]), y] = 0
    loss = np.sum(margins) / X.shape[0]
    loss += 0.5 * alpha * np.sum(W * W)
    binary_mask = (margins > 0).astype(float)
    binary_mask[np.arange(X.shape[0]), y] -= np.sum(binary_mask, axis=1)
    dW = X.T.dot(binary_mask) / X.shape[0]
    dW += alpha * W
    return loss, dW
    ###End solution

In [17]:
# NO REGLARIZATION
loss, dW = alternative_classification_loss_vectorized(W, X, y, 0.0)

f = lambda W: alternative_classification_loss_vectorized(W, X, y, 0.0)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-9)

numerical: -0.502000 analytic: -0.502000, relative error: 2.215581e-12
numerical: 0.083333 analytic: 0.083333, relative error: 6.989051e-11
numerical: -0.370667 analytic: -0.370667, relative error: 8.587256e-12
numerical: 0.287333 analytic: 0.287333, relative error: 1.202605e-11
numerical: -0.126667 analytic: -0.126667, relative error: 8.788047e-11
numerical: 0.083333 analytic: 0.083333, relative error: 6.989051e-11
numerical: -0.744667 analytic: -0.744667, relative error: 1.263776e-11
numerical: 0.287333 analytic: 0.287333, relative error: 1.202605e-11
numerical: 0.837333 analytic: 0.837333, relative error: 2.002379e-12
numerical: -0.370667 analytic: -0.370667, relative error: 8.587256e-12
numerical: -0.092667 analytic: -0.092667, relative error: 2.355797e-11
numerical: -0.092667 analytic: -0.092667, relative error: 2.355797e-11


In [18]:
# REGLARIZATION
loss, dW = alternative_classification_loss_vectorized(W, X, y, 2)

f = lambda W: alternative_classification_loss_vectorized(W, X, y, 2)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-9)

numerical: 2.295972 analytic: 2.295972, relative error: 6.223611e-12
numerical: -0.744774 analytic: -0.744774, relative error: 1.751270e-11
numerical: 0.837413 analytic: 0.837413, relative error: 5.972695e-12
numerical: -0.092829 analytic: -0.092829, relative error: 5.426796e-11
numerical: 2.295972 analytic: 2.295972, relative error: 6.223611e-12
numerical: -0.744774 analytic: -0.744774, relative error: 1.751270e-11
numerical: -0.744774 analytic: -0.744774, relative error: 1.751270e-11
numerical: 0.287391 analytic: 0.287391, relative error: 1.013313e-11
numerical: 0.837413 analytic: 0.837413, relative error: 5.972695e-12
numerical: -0.092829 analytic: -0.092829, relative error: 5.426796e-11
numerical: -0.826432 analytic: -0.826432, relative error: 5.246490e-12
numerical: 0.952759 analytic: 0.952759, relative error: 1.940992e-12


In [19]:
class LinearModelClassification():
    def __init__(self, fit_intercept=True):
        self.W = None
        self.fit_intercept = fit_intercept

    def train(self, X, y, learning_rate=1e-3, alpha=0, num_iters=100, batch_size=200, verbose=False):
        if self.fit_intercept:
            ### Begin solution
            X = np.c_[np.ones(X.shape[0]), X]
            ### End solution
            
        N, d = X.shape
        
        C = (np.max(y) + 1) 
        if self.W is None: # Initialization
            self.W = 0.001 * np.random.randn(d, C)

        # Run stochastic gradient descent to optimize W
        
        loss_history = []
        for it in range(num_iters):
            X_batch = None
            y_batch = None
                                                               
            # Sample batch_size elements in X_batch and y_batch
            indices = np.random.choice(N, batch_size)
            X_batch = X[indices]
            y_batch = y[indices]
            
            # evaluate loss and gradient
            loss, dW = self.loss(X_batch, y_batch, alpha)
            loss_history.append(loss)

            # perform parameter update                                                                
            # Update the weights w using the gradient and the learning rate.          
            ### Begin solution
            self.W -= learning_rate * dW
            ### End solution
            
            if verbose and it % 10000 == 0:
                print("iteration %d / %d: loss %f" % (it, num_iters, loss))
                
        return loss_history

    def predict(self, X):
        pass

    def loss(self, X_batch, y_batch, reg):
        pass

class AlternativeClassificationModel(LinearModelClassification):
    """ Multiclass classification model """

    def loss(self, X_batch, y_batch, alpha):
        return alternative_classification_loss_vectorized(self.W, X_batch, y_batch, alpha)
    
    def predict(self, X):
        """ 
        Inputs:
        - X: array of shape (N, D) 

        Returns:
        - y_pred: 1-dimensional array of length N, each element is an integer giving the predicted class 
        """
        ### Begin solution
        if self.fit_intercept:
            X = np.c_[np.ones(X.shape[0]), X]
        scores = X.dot(self.W)
        y_pred = np.argmax(scores, axis=1)
        return y_pred
        ### End solution

In [20]:
model = AlternativeClassificationModel()
model.train(X, y, num_iters=75000, batch_size=64, learning_rate=1e-3, verbose=True)
pred = model.predict(X)
model_accuracy = accuracy_score(y, pred)
print(model_accuracy)
assert model_accuracy > 0.97

iteration 0 / 75000: loss 1.993898
iteration 10000 / 75000: loss 0.110852
iteration 20000 / 75000: loss 0.135989
iteration 30000 / 75000: loss 0.055490
iteration 40000 / 75000: loss 0.077591
iteration 50000 / 75000: loss 0.105372
iteration 60000 / 75000: loss 0.033110
iteration 70000 / 75000: loss 0.178649
0.9733333333333334


# Logistic regression with CVXPY

In [21]:
data2 = load_breast_cancer()
X2, y2 = data2.data, data2.target
scaler = StandardScaler()
X2 = scaler.fit_transform(X2)

In [22]:
def sigmoid(z):
    return 1/(1 + np.exp(-z))

class LogisticRegressionCVXPY():
    def __init__(self, fit_intercept=True, alpha=1.0):
        self.w = 0
        self.fit_intercept = fit_intercept # bias
        self.alpha = alpha
    
    def fit(self, X, y):
        ### Begin solution
        m, n = X.shape
        self.w = cp.Variable(n + 1) if self.fit_intercept else cp.Variable(n)
        X_w = X @ self.w[1:] + self.w[0] if self.fit_intercept else X @ self.w
        log_likelihood = cp.sum(cp.multiply(y, X_w) - cp.logistic(X_w))
        regularization_term = 0.5 * self.alpha * cp.sum_squares(self.w[1:])
        objective = cp.Maximize(log_likelihood - regularization_term)
        constraints = []
        problem = cp.Problem(objective, constraints)
        problem.solve()
        self.w = self.w.value
        ### End solution

    def predict(self, X):
        """ Return prediction labels vector of 0 or 1 """
        ### Begin solution
        if self.fit_intercept:
            return np.round(sigmoid(X @ self.w[1:] + self.w[0]))
        else:
            return np.round(sigmoid(X @ self.w))
        ### End solution

### without bias

In [23]:
model = LogisticRegressionCVXPY(alpha=1e-3, fit_intercept=False)
model.fit(X2, y2)
pred = model.predict(X2)
accuracy = accuracy_score(y2, pred)
print(accuracy)
assert accuracy >= 0.98

0.9929701230228472


### with bias

In [24]:
model = LogisticRegressionCVXPY(alpha=1e-3, fit_intercept=True)
model.fit(X2, y2)
pred = model.predict(X2)
accuracy = accuracy_score(y2, pred)
print(accuracy)
assert accuracy >= 0.99

0.9947275922671354


# Alternative Binary classification CVXPY
 
Implement a binary classification model with CVXPY whose parameters are obtained by: (label is 1 and -1 instead of 0 and 1)

$$\min_{\mathbf{w},b}\frac{1}{2}||\mathbf{w}||^2$$ <br>
$$\text{s.t } y_i(\mathbf{w}^{\top}\mathbf{x}_i + b) \ge 1, \ i=1...N$$

In [25]:
X3, y3 = make_blobs(n_samples=300, centers=2, n_features=12, random_state=47)
scaler = StandardScaler()
X3 = scaler.fit_transform(X3)
y3[y3 == 0] = -1

In [26]:
class BinaryClassificationModel():
    def __init__(self):
        self.w = None
        self.b = 0
    
    def fit(self, X, y):
        ### Begin solution 
        N, d = X.shape
        w = cp.Variable(d)
        b = cp.Variable()
        objective = cp.Minimize(0.5 * cp.norm(w)**2)
        constraints = [y[i] * (X[i] @ w + b) >= 1 for i in range(N)]
        problem = cp.Problem(objective, constraints)
        problem.solve()
        self.w = w.value
        self.b = b.value
        ### End solution
        
    def predict(self, X):
        """Return the predicted label 1 or -1"""
        y_pred = X @ self.w + self.b
        return np.sign(y_pred)

In [27]:
model = BinaryClassificationModel()
model.fit(X3, y3)
pred = model.predict(X3)
accuracy = accuracy_score(y3, pred)
print(accuracy)
assert accuracy == 1

1.0


# Alternative  Binary classification 2 CVXPY
 
Implement a binary classification model with CVXPY whose parameters minimize the loss

$$L(\mathbf{w},b) = \frac{1}{N} \sum_{i=1}^N \max(0, y_i(\mathbf{w}^{\top}\mathbf{x}_i + b)) + \lambda||\mathbf{w}||^2_2$$

In [28]:
data4 = load_breast_cancer()
X4, y4 = data4.data, data4.target
scaler = StandardScaler()
X4 = scaler.fit_transform(X4)
y4[y4 == 0] = -1

In [29]:
class BinaryClassificationModel2():
    def __init__(self, alpha=0):
        self.w = None
        self.b = 0
        self.alpha = alpha
    
    def fit(self, X, y):
        ### Begin solution
        N, d = X.shape
        w = cp.Variable(d)
        b = cp.Variable()
        loss = cp.sum(cp.maximum(0, 1 - cp.multiply(y, X @ w + b))) / N + self.alpha * cp.norm(w, 2)**2
        objective = cp.Minimize(loss)
        problem = cp.Problem(objective)
        problem.solve()
        self.w = w.value
        self.b = b.value
        ### End solution 
        
    def predict(self, X):
        """Return the predicted label 1 or -1""" 
        y_pred = X @ self.w + self.b
        return np.sign(y_pred)

In [30]:
model = BinaryClassificationModel2(alpha=1e-3)
model.fit(X4, y4)
pred = model.predict(X4)
accuracy = accuracy_score(y4, pred)
print(accuracy)
assert accuracy >= 0.98

0.9876977152899824


# Lasso with gradient descent

In [31]:
data = load_diabetes()
X5, y5 = data.data, data.target

def mse_loss_vectorized(w, b, X, y):
    """
    MSE loss function WITHOUT FOR LOOPs , NO REGULARIZATION
    
    Returns a tuple of:
    - loss 
    - gradient with respect to weights w
    - gradient with respect to bias b
    """
    loss = 0.0
    dw = np.zeros_like(w)
    
    loss = np.mean(np.square(X @ w + b - y)) 
    dw = ((X.T @ (X @ w + b - y)) / X.shape[0])
    db =  np.sum(X @ w + b - y) / X.shape[0]
    
    return loss, dw, np.array(db).reshape(1,)

In [32]:
def lasso_gradient_mse_loss_vectorized(w, b, X, y, alpha):
    """
    MSE loss function adding the subgradient for w
    """
    loss, dw, db = mse_loss_vectorized(w, b, X, y)
    ### Begin solution
    l1_regularization = alpha * np.sum(np.abs(w))
    loss += l1_regularization
    subgradient = alpha * np.sign(w)
    dw += subgradient
    ### End solution
    return loss, dw, db

In [33]:
class LassoGradientDescent():
    def __init__(self,  alpha=0.1):
        self.w = None
        self.b = None
        self.alpha = alpha

    def train(self, X, y, learning_rate=1e-3, num_iters=100, batch_size=200, verbose=False):
        N, d = X.shape
        
        if self.w is None: # Initialization
            self.w = 0.001 * np.random.randn(d)
            self.b = 0.0

        # Run stochastic gradient descent to optimize w
        
        loss_history = []
        for it in range(num_iters):
            X_batch = None
            y_batch = None
                                                               
            # Sample batch_size elements in X_batch and y_batch
            indices = np.random.choice(N, batch_size)
            X_batch = X[indices]
            y_batch = y[indices]
            
            # evaluate loss and gradient
            loss, dw, db = self.loss(X_batch, y_batch)

            # perform parameter update                                                                
            # Update the weights w using the gradient and the learning rate.  
            ### Begin solution
            self.w = self.w - learning_rate * dw
            self.b = self.b - learning_rate * db
            ### End solution
            
            if verbose and it % 10000 == 0:
                print("iteration %d / %d: loss %f" % (it, num_iters, loss))
    
    
    def predict(self, X):
        ### Begin solution
        return X @ self.w + self.b
        ### End solution

    def loss(self, X_batch, y_batch):
        return lasso_gradient_mse_loss_vectorized(self.w, self.b, X_batch, y_batch, self.alpha)

In [34]:
model = LassoGradientDescent(alpha=0.1)
model.train(X5, y5, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X5)
mse = mean_squared_error(pred, y5)

sk_model = Lasso(alpha=0.1, fit_intercept=True)
sk_model.fit(X5, y5)
sk_pred = sk_model.predict(X5)
sk_mse = mean_squared_error(sk_pred, y5)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

iteration 0 / 200000: loss 27574.957344
iteration 10000 / 200000: loss 3469.001199
iteration 20000 / 200000: loss 3823.674676
iteration 30000 / 200000: loss 3336.732402
iteration 40000 / 200000: loss 3089.978700
iteration 50000 / 200000: loss 3391.335442
iteration 60000 / 200000: loss 2874.821783
iteration 70000 / 200000: loss 3408.671554
iteration 80000 / 200000: loss 3431.741126
iteration 90000 / 200000: loss 2867.556888
iteration 100000 / 200000: loss 3254.812423
iteration 110000 / 200000: loss 3396.375976
iteration 120000 / 200000: loss 3141.754995
iteration 130000 / 200000: loss 3259.679814
iteration 140000 / 200000: loss 2998.965199
iteration 150000 / 200000: loss 3127.847574
iteration 160000 / 200000: loss 3114.553723
iteration 170000 / 200000: loss 3049.987563
iteration 180000 / 200000: loss 3330.985338
iteration 190000 / 200000: loss 3010.846620
MSE scikit-learn: 2912.52749260362
MSE Coordinate descent model : 2916.2262375671194


In [35]:
model = LassoGradientDescent(alpha=2)
model.train(X5, y5, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X5)
mse = mean_squared_error(pred, y5)

sk_model = Lasso(alpha=2, fit_intercept=True)
sk_model.fit(X5, y5)
sk_pred = sk_model.predict(X5)
sk_mse = mean_squared_error(sk_pred, y5)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

iteration 0 / 200000: loss 27028.442014
iteration 10000 / 200000: loss 5648.962726
iteration 20000 / 200000: loss 5727.423210
iteration 30000 / 200000: loss 7207.600220
iteration 40000 / 200000: loss 5737.715557
iteration 50000 / 200000: loss 6119.368381
iteration 60000 / 200000: loss 5296.933341
iteration 70000 / 200000: loss 5762.981584
iteration 80000 / 200000: loss 5182.832004
iteration 90000 / 200000: loss 5365.864171
iteration 100000 / 200000: loss 5875.605303
iteration 110000 / 200000: loss 5229.014975
iteration 120000 / 200000: loss 5589.115935
iteration 130000 / 200000: loss 5814.025776
iteration 140000 / 200000: loss 5785.417651
iteration 150000 / 200000: loss 6504.894675
iteration 160000 / 200000: loss 6108.755464
iteration 170000 / 200000: loss 6196.674392
iteration 180000 / 200000: loss 6083.212100
iteration 190000 / 200000: loss 5953.898235
MSE scikit-learn: 5650.290772564548
MSE Coordinate descent model : 5653.150193099002
